# 90 — Validate (train → OOS, metrics report)

Full pipeline: `azlib.validate.run_train` on the ambient TRAIN dataset,
`azlib.validate.run_oos` on a SEPARATE `oos_env` (a real env-FILE PATH
parameter — see `nb/README.md`'s two run modes), then
`azlib.validate.metrics` for the design spec §8 success metrics
(`strict_coverage`, `non_strict_coverage`, `realized_rr`,
`reach_drift_max`). All three are already-tested `azlib.validate`
functions (Task 8) — this notebook only wires them together, persists the
OOS zoned dataset, and writes the metrics report.

`oos_env` defaults to the real 2-month OOS config (`configs/oos2m_dataset
.env`) for the documented full-2y run (see `nb/README.md`); the automated
smoke gate overrides it with `-p oos_env <smoke oos env>`, never the real
dataset.

In [ ]:
tf = 15
direction = "long"
oos_env = "configs/oos2m_dataset.env"

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebooks/action_zones/{azlib,nb} onto sys.path. Assumes cwd is
# the repo root (/code) -- true for the experiment Docker service
# (docker-compose.experiment.yml's working_dir: /code; papermill's own
# cwd=None default inherits the invoking process's cwd) and for the
# automated smoke test (tests/test_layer9_notebooks.py sets the papermill
# subprocess's cwd to the repo root to match). helpers.py/indicators.*
# resolve for free via the image's own PYTHONPATH=/code.
_AZ_ROOT = Path.cwd() / "notebooks" / "action_zones"
for _p in (_AZ_ROOT, _AZ_ROOT / "nb"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import nbutil

In [ ]:
import tempfile

from azlib.validate import run_train, run_oos, metrics
from azlib.zones import ResultsFile

In [ ]:
train_env = nbutil.snapshot_env_file(tempfile.mkstemp(suffix=".env")[1])

results_path = run_train(train_env, tf, direction)

# out_dir MUST be resolved here, before run_oos -- run_oos internally calls
# azlib.validate._apply_env_file(oos_env), which MUTATES os.environ to the
# OOS dataset's own ROOT_FOLDER/DATA_ROOT/DATA_SET_NAME/PAIR. Computing
# out_dir (env-derived, via helpers.dataset_folder()) any later would
# silently resolve it against the OOS dataset dir instead of train's.
out_dir = nbutil.results_dir(tf, direction, "validate")

zoned_oos = run_oos(results_path, oos_env)

rf = ResultsFile.load(results_path)
m = metrics(zoned_oos, tf, direction, label_params=rf.label_params)
print(m)

In [ ]:
zoned_dataset_path = f"{out_dir}/zoned_dataset_oos.pkl"
zoned_oos.to_pickle(zoned_dataset_path)

report = {
    "tf": tf,
    "direction": direction,
    "results_path": results_path,
    "oos_env": oos_env,
    "zoned_dataset_path": zoned_dataset_path,
    "n_rows": len(zoned_oos),
    "metrics": m,
}
report_path = nbutil.write_json_report(f"{out_dir}/metrics_report.json", report)
print(report_path)